In [36]:
from tdc.single_pred import ADME
data = ADME(name='Solubility_AqSolDB')
split = data.get_split(method = 'scaffold')

train = split['train']
print(train.columns.tolist())
print(train.head())
print({k: len(v) for k, v in split.items()})
print(train['Y'].describe())

#Y is the log solubility, Continuous - Most durg-like molecules dissolve at concentrataions below 1 hence log < 1
#The sign is just saying "less than one mole per litre" solubility expressed in mol/L
#More negative means less soluble.


Found local copy...
Loading...
Done!
100%|██████████| 9982/9982 [00:01<00:00, 9064.99it/s] 

['Drug_ID', 'Drug', 'Y']
                                             Drug_ID  \
0                               4-chlorobenzaldehyde   
1                                       vinyltoluene   
2                      4-(dimethylamino)benzaldehyde   
3               2-methyl-1-phenylpropan-2-yl acetate   
4  5-methoxy-1-[4-(trifluoromethyl)phenyl]pentan-...   

                            Drug         Y  
0                O=Cc1ccc(Cl)cc1 -2.177078  
1                 C=Cc1cccc(C)c1 -3.123150  
2             CN(C)c1ccc(C=O)cc1 -2.282769  
3        CC(=O)OC(C)(C)Cc1ccccc1 -2.394650  
4  COCCCCC(=O)c1ccc(C(F)(F)F)cc1 -3.544060  
{'train': 6987, 'valid': 998, 'test': 1997}
count    6987.000000
mean       -2.578511
std         2.251413
min       -11.998938
25%        -3.950000
50%        -2.349000
75%        -0.962413
max         1.967513
Name: Y, dtype: float64


In [37]:
#split.items()- split is the dictionary, calling .items() on a dict gives its key-value pairs one at a time
#k here is the key, dict. made of key-value pairs so the comprehension has to produce two things on every pass

print({k: len(v) for k, v in split.items()})

{'train': 6987, 'valid': 998, 'test': 1997}


In [38]:
#df gets the value (the DataFrame). On each pass
for name, df in split.items():
    print(name, round(df['Y'].mean(), 2), round(df['Y'].std(), 2))


#Means have spread apart, Under a random split they would be much closer/near identical 
#Valid and test are noticeable more negative meaning the held-out scaffolds are, on average less soluble than the training ones
#std also spreads out menaing valid.  set covers a somewhat broarder range of solublilities

train -2.58 2.25
valid -4.04 2.75
test -3.4 2.3


In [39]:
print(train['Y'].describe())
# train is a pandas Dataframe (a table). Indexing it with ['Y'] pulls out a single column by name. 
# .describe() - pandas method that computes a batch of summary stats for that given pandas series.

count    6987.000000
mean       -2.578511
std         2.251413
min       -11.998938
25%        -3.950000
50%        -2.349000
75%        -0.962413
max         1.967513
Name: Y, dtype: float64


In [40]:
from rdkit import Chem # Cheminformatics library, Chem is the core module

#MolFromSmiles parses the SMILES and builds a proper Mol object - RDKit's internal representation of the molecules 
#Structure, with actual atom objects and bond objects connecting them. This Parse step is the bridge from "text" to "Chemistry that can be queried"

mol = Chem.MolFromSmiles(train['Drug'].iloc[0]) #Grabs the Drug column (smiles string) as a series 
#.iloc[0] - iloc means "index by integer position" and [0] takes the first one, hence pulls SMILES string of the first molecule in the training set

print("atoms",mol.GetNumAtoms()) #now that mol is a real structure, can ask quesions - like "how many atoms"

atoms 9


In [41]:
#mol is a structured object, RDKit parsed the SMILES text and build a Mol object in memory. It's a container that holds, among other things a collection of atom objects and bond objects
# and information about how they are connected.
#mol.GetAtoms() is a method on that object - a function attatched to mol that, when called hands back the collection of atom objects living inside it. 
#It returns somthing you can iterate over, where ech item is a full Atom object, not a letter, a object that itself has methods (e.g GetSymbol(), GetDegree(),etc)
#Pattern: mol.GetSomthing() - asks the whole molecule a question (e.g GetNumAtoms()), atom.GetSomthing() asks a single atom a question (GetSymbol(),GetDegree())


for atom in mol.GetAtoms(): 
    print(atom.GetSymbol(),atom.GetDegree())

O 1
C 2
C 3
C 2
C 2
C 3
Cl 1
C 2
C 2


In [42]:
def atom_features(atom): #atom in the parenthisis as atom is the parameter - this function needs one input to d its job and i'll call that input "atom"
   #atom_features isn't about one specific atom - its a general procedure for turning any atom into 14 numbers. It needs a slot to receive whichever atom you want processed. That slot is the parameter. 
    allowed = ['C','N','O','F','P','S','Cl','Br','I']
# One-hot slice (length 10: 9 elements + 1 "other")
    symbol = atom.GetSymbol()
    onehot = [0] * (len(allowed) + 1) # Ten zeros 
    if symbol in allowed:
        onehot[allowed.index(symbol)] = 1
    else:
        onehot[-1] = 1 #The "other" bucket. [-1] is the last slot. Any element not on the list lands here producing an al zero-slice. 

    degree = atom.GetDegree()
    charge = atom.GetFormalCharge()
    aromatic = int(atom.GetIsAromatic()) #GetIsAromatic returns a Python True/False, not a number. int(True) = 1
    num_hs = atom.GetTotalNumHs() # adding () calls the function, actually running it 

    return onehot + [degree,charge,aromatic,num_hs]


In [43]:
first_atom = mol.GetAtoms()[0] 
print(atom_features(first_atom))
print(len(atom_features(first_atom)))

[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
14


In [44]:
atom_rows = [atom_features(atom) for atom in mol.GetAtoms()]
#Loop over every atom in the molecule, run the featuriser on each, collect the results into a list. 
#So atoms_rows becomes a list of 9 lists, each of length 14, exactly the matrix shape wanted.
#For atom in mol.GetAtoms() - loops over the molecule's atoms; each loop, the current atom object gets the name atom
#atom_features(atom) - this takes the current atom and passes it into the function
#on each pass, a real atom object flows into the function's placeholder, comes back as 14 numbers an dgets collected into the list. Nine atoms oin -> nine out

#in def atom_features(atom) - atom is a placeholder naminng the input the function expects 
#in atom_features(atom) (when calling) atom is the actual atom you're feeding in

In [45]:
print(len(atom_rows))
print(len(atom_rows[0]))

9
14


In [46]:
#right now atom_rows is a python list of lists. PyTorch doesn't train on plain lists - it needs a tensor (its own numeric array type)

import torch 
x = torch.tensor(atom_rows, dtype=torch.float)
print(x.shape)

#dtype=torch.float - the network does floating-point math, so we declare these floats (not integers)
#x.shape - tensor knows its own dimensions. seeing that shape is prrof the whole atom-featurization pipeline works end to end 
x

torch.Size([9, 14])


tensor([[0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 0., 0., 1.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 0., 1., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 0., 1., 1.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 0., 1., 1.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 3., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 0., 1., 1.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 2., 0., 1., 1.]])

In [47]:
for bond in mol.GetBonds(): #returns a collection of bond objects inside the molecule, just like each atom was a rich Atom object.
    #The loop runs once per chemical bond in the molecule. A bond object knows things like its type(single/double/aromatic) and the part needed, how many atoms it connects
    i = bond.GetBeginAtomIdx()
    j = bond.GetEndAtomIdx()
    print(i,j)

    #mol.GetBonds() - molecule's bonds
    #GetBeginAtomIdx() and GetEndAtomIdx() - the integer indicies of the two atoms a bond connects
    #Indicies, not atom objects, which is what edge_index wants, since it referes to atoms by their row bumber in x

    
#A bond, in this representation is defined as a connection between exactly two atoms. This is the property of the single bond 
#RDKit arbitrarily labels one end the "begin" atom and the other the "end" atom (bond is undirected, this is just arbritrary). GetBeginAtomIdx() returns the index of the begin atom
#Returns an integer, the atom's position number in the molecule. SO if the bond connects the 1st ad 2nd aaotms, i might be 0. This integer is exactly what edge_index needs
#edge_index refers to atoms by thei row number in your x matric, not by the atom objects themselves. 
#the atoms are identifies by their row number. So when a bond says " I connect atom i to atom j", those i and j are pointing at rows in the feature matric
#A bond [0,1] means row 0 of x is bonded to row 1 of x
#x says what each atom is, and edge_index says which rows are connected. 

0 1
1 2
2 3
3 4
4 5
5 6
5 7
7 8
8 2


In [48]:
import torch 

edge_list = []
for bond in mol.GetBonds():
    i = bond.GetBeginAtomIdx()
    j = bond.GetEndAtomIdx()
    edge_list.append([i,j])
    edge_list.append([j,i])

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous() #.t() transpose flipped the list of pairs into rows 
print(edge_index.shape)

#torch.Size([2,18]) is describing the dimensions of the edge_index tensor - its shape, not its contents. It reads 2 rows, 18 columns. 
#2 rows because every edge needs exactly two pieces of information: a source atom and a destination atom.Row 0 holds all the sources; row 1 holds all the destinations
#18 columns - becasue there is 18 directed edges, this is 9 bonds x2 directions. Each column is one directed edge.

torch.Size([2, 18])


In [49]:
edge_index
#each column is one directed edge, top to bottom = source -> destination
#displaying as columns is much mrore efficient, message passing grabs all the source atoms and all the destination atoms, storing them as two long rows makes that a single fast slice
#edge_index[0] = every source, edge_index[1] = every destination 
#the numbers in the tensor are the same indices as rows of the matrix x. so when column 0 says 0->1 it means "row 0 of x" doesn't store atoms - it stores pointers into x.


tensor([[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 5, 7, 7, 8, 8, 2],
        [1, 0, 2, 1, 3, 2, 4, 3, 5, 4, 6, 5, 7, 5, 8, 7, 2, 8]])

In [50]:
from torch_geometric.data import Data

In [ ]:
#y is the target - the answer that the network is trying to predict. Every training example needs two parts: the input (structure of molecule, x + edge_index) and the correct output
#Correct output (the actual solubility) - which is y. During the training the network will make a guess for this molecules and then compare it against y to see how wrong it was


#train['Y'] selects the Y column of the DataFrame ( whole column of solubility values, one per molecule) as a pandas series
#train['Y'].iloc[0] - .iloc[0] takes the value at position 0 - the first molecule's solubility. Crutially it is the same molecule 0 that x and edege_index descibe - so the input and the label match up
#The square brackets around it. Wrapping it in [] puts it inside a lise of one element.

y = torch.tensor([train['Y'].iloc[0]], dtype=torch.float) #wraps teh value into a PyTorch tensor, the numeric array type the network operates on, .float declares it as a floating-point number



data = Data(x=x, edge_index=edge_index,y=y)
print(data)


Data(x=[9, 14], edge_index=[2, 18], y=[1])


In [53]:
def smiles_to_graph(smiles,target):
    mol = Chem.MolFromSmiles(smiles)

    #--- node features (this is on mol) ---
    atom_rows = [atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(atom_rows, dtype=torch.float)

    # --- edges --- 
    edge_list = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_list.append([i,j])
        edge_list.append([j,i])
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

    # --- target ---
    y = torch.tensor([target],dtype=torch.float)

    return Data(x=x, edge_index=edge_index, y=y)

In [55]:
g = smiles_to_graph(train['Drug'].iloc[0], train['Y'].iloc[0])
print(g)

Data(x=[9, 14], edge_index=[2, 18], y=[1])


In [64]:
#Will run smiles_to_graph on every molecule in the training set and collect the results into a list of Data objects. That list is the dataset
#MolFromSmiles returns None for any SMILES it can't parse, and real datasets contain a few bad ones. If you call your function on a None mol. mol.GetAtoms() crashes, hence the outerloop needs to skip unparseable molecules rather than die on them 

def make_dataset(df):
    graphs = [] #empty list
    for smiles, target in zip(df['Drug'], df['Y']): #This is a way to loop over two columns in lockstep, zip pairs them up, keeps each molecule structure and its solubilty together.
        mol = Chem.MolFromSmiles(smiles)
        if mol is None: #This is the bad SMILES guard, so unparseable SMILES get silently passed over instead of crashing the whole run.
            continue
        graph = smiles_to_graph(smiles,target) # <- build ONE data object
        graphs.append(graph) # <- add that Data object 
    return graphs

#A graph here is a Data object - one complete molecule converted into the atoms-plus-bonds-plus-target structure. Its the Data(x=[9, 14], edge_index=[2, 18], y=[1])
#A molecule in graph form!
# The thing being appendd is graph - which is the output of smiles_to_graph, i.e a fully build Data object. Not the SMILES string.
#The SMILES string and the target are getting consumed to build each graph, after the loop finishes, graphs is a list of Data objects, each entry being a molecule


In [ ]:
train_graphs = make_dataset(train)
print(len(train_graphs))
print(train_graphs[0])

#This is the full training set as list of graphs

6987
Data(x=[9, 14], edge_index=[2, 18], y=[1])


In [67]:
valid_graphs = make_dataset(split['valid'])
test_graphs = make_dataset(split['test'])

print(len(train_graphs), len(valid_graphs), len(test_graphs))

6987 998 1997


In [ ]:
#Batching, a GNN is fed in batches for effient, stable training. PyG has DataLoader; it takes batches of separate molecular graphs and merges them into one big disconnected graph, so the GNN can process them together. 

from torch_geometric.loader import DataLoader
train_loader = DataLoader(train_graphs, batch_size=64, shuffle = True)
valid_loader = DataLoader(valid_graphs, batch_size=64, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=64, shuffle=False)

#Shuffle=True for train, False for valid/test - training data is shuffled each epoch so the model doesn't learn anything from the order of molecules, but there's no reason to shuffle evaluation sets
#Keeping evaluation sets unshuffled keeps them fixed ad makes results reproducible

batch = next(iter(train_loader))
print(batch)

#Output contains 64 molecules merged into one big structure
#x=[1024, 14]- the node features the whole batch, theres 1024 atoms across all 64 molecules, each described by the 14 features. PyG stacked all 64 molecules' atoms into one small tall matrix
#edge_index=[2,2038]all the bonds from the 64 molecules, same [2, num_edges] fromat as one molecule, just bigger. 
#batch=[1024] This is a vector with one entry per atom labelling which molecule each atom belongs to. The first molecules atoms are flagged as 0, the next molecule's tagged as 1 and so on
#After the GNN processes all 1024 atoms, you need to collapse each molecule's atoms back into one prediction per molecule. The batch vector is the map that says "these atoms belong together"
#ptr=[65] - pointers markeing where each molecule starts/ends in that stacked x. 65 entires for 64 molecules (boundaries need one extra marker)

git


DataBatch(x=[1024, 14], edge_index=[2, 2038], y=[64], batch=[1024], ptr=[65])


In [ ]:
#Now for the GNN itself. There is 3 stages
# --- 1 - Message-passing layers - each atom repeatedly gathers infromation from its bonded neighbours, so its vector comes to encode its local chemical enviornment
# --- 2 - Readout/pooling - collapse all of a molecule's atom vectors into one molecule-level vector. This is where that batch vector gets used: global_mean_pool averages each molecule's atoms together. Many atoms -> one molecule fingerprint
# --- 3 - Prediction head - a small ordinary neural network that takes the molecule vector and outputs a single number: the predicted solubility
# So the flow is: atoms gather from neighbours (xa few rounds)-> pool atoms into one molecule vector -> predict one number

#Every PyTorch model has exactly two parts
# --- __init__ - defines the layers (the pieces, created once)
# --- forward - defines the flow (how data passes through the pieces)

import torch
import torch.nn.functional as F # nickname so can type F.relu instead of torch.nn.functional.relu every time
from torch_geometric.nn import GCNConv, global_mean_pool 
#GCNConv - a message-passing layer (the "atoms gather from neighbours" stage)
#global_mean_pool the pooling function (the "collapse atoms into one molecule vector" stage)
from torch.nn import Linear #Linear - an ordinary neural-network layer ( the "make the final prediction" stage)

#Function - data goes in, result comes out, nothing is remembered. But a NN needs to remember somthing between uses; its learnable weights - the number it adjusts as it learns
#The weights have to persist and be reused every time you run the model. A class is Python's tool for bundlin data that persists (the layers/weights) together with behavious (how to run them)

#So the class packages two things:
# --- The layers (which hold the weights) - set up once, remembered 
# --- The recipe for pushing data through those layers - run many times


class GNN(torch.nn.Module): # "Defining a new model called GNN" the torch.nn.Module part means it inherits from PyTorch's base model class, Module.
    #Inheritance means 'GNN is a kind of Module, and automatically gets all of Module's built-in machinery for free" - things like tracking weights, moving to GPU, saving/loading.
    #Every PyTorch model inherits from MOdule; it's mandatory 



# __init__ = what pieces exist, forward = how data flows through them
    def __init__(self, num_features, hidden_dim): #Takes two settings, num_features (14 in this case) and hidden_dim (how wide the internal vectors should be - 64) - allow the configuration of the model's size 
#seld - refers to this particular model object - kind of like the models backpack, put the layers in during setup, and pull them out during the forward pass
        super().__init__() # This runs the setup of the parent Module class, switching on all the inherited weight-tracking machinery. Always the first line of a PyTorch model

# The 3 layers 

# Stage 1: message-passing layers
# Takes 14-dimensional atom vectors and outputs 64-dimensional ones. Every layer works this way: (input_size, output_size) Widens each atom from 14 numbers to 64, while also mixing in neighbour information
# This just makes the information "richer" of each molecule
        self.conv1 = GCNConv(num_features, hidden_dim)
        
#Stage 2: A second pasing round: 64 inn, 64 out. ONe GCNConv lets an atom see its direct neighbout; a second lets it see neighbours-of-neighbours. Stacking = wider awareness

        self.conv2 = GCNConv(hidden_dim, hidden_dim)
# Stage 3: prediction head. Takes a 64-dimensional molecule vector and crushes it down to 1 number, the predicted solubility
        self.lin = Linear(hidden_dim, 1)





# this is where the data actually flows. 
    def forward(self, x, edge_index, batch):
        # Stage 1: message passing, with non-linearity between layers
        #The three arguments are exactly the three things sitting in the DataBatch: x (atom features), edge_index(bonds), batch (which atom belongs to which molecule).
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        # Stage 2: readout — pool atoms into one vector per molecule
        x = global_mean_pool(x, batch)
        # Stage 3: predict one number
        x = self.lin(x)
        return x

In [80]:
model = GNN(num_features=14, hidden_dim=64)
print(model)

GNN(
  (conv1): GCNConv(14, 64)
  (conv2): GCNConv(64, 64)
  (lin): Linear(in_features=64, out_features=1, bias=True)
)
